**This notebook is used to download all of the character pages from the fandom.wiki pages of Lord of the Rings and Harry Potter**

### Download data

In [1]:
#Import all relevant libraries
import networkx as nx
import re, pickle, urllib.request, time, json

In [2]:
#Define helper functions for data download
def download_pages(url, page_names, batch_size=25):
    api_url = f'https://{url}/api.php'
    results = []
    batches = range(0, len(page_names), batch_size)
    N = len(batches)
    print(f'Total number of batches: {N}. Total number of pages: {len(page_names)}')
    failed_batches = []

    for i in batches:
        batch = page_names[i:i+batch_size]
        clean_batch = [t for t in batch if t and not any(c in t for c in '\n\r{}')]
        if not clean_batch:
            continue

        data = {
            'action': 'query',
            'prop': 'revisions',
            'rvprop': 'content',
            'titles': '|'.join(clean_batch),
            'format': 'json'
        }

        encoded = urllib.parse.urlencode(data).encode('utf-8')
        request = urllib.request.Request(api_url, data=encoded)
        request.add_header('User-Agent', 'YourAppName/1.0 (s204229@student.dtu.dk)')

        try:
            with urllib.request.urlopen(request) as response:
                wikidata = json.loads(response.read())
                for key, page in wikidata['query']['pages'].items():
                    revisions = page.get('revisions', [{}])
                    content = revisions[0].get('*', '') if revisions else ''
                    results.append({
                        'page_name': page.get('title', ''),
                        'content': content
                    })
        except urllib.error.HTTPError as e:
            print(f"HTTP Error {e.code} on batch {i//batch_size + 1} / {N}: {e.reason}")
            failed_batches += batch
        time.sleep(1)
    return results, failed_batches

def get_all_characters(url, root_category):
    downloaded_categories = [root_category]
    response = get_category_members(url, root_category)
    characters, categories = extract_characters_and_categories(response, downloaded_categories)

    while categories:
        current = categories.pop(0)
        downloaded_categories.append(current)
        print(f"{len(categories)} remaining. Fetching: {current}")
        response = get_category_members(url, current)
        char, cat = extract_characters_and_categories(response, downloaded_categories)
        characters = list(set(characters + char))
        categories = list(set(categories + cat) )
        time.sleep(1)

    return sorted(set(characters))

def get_category_members(url, page_name):
    base_url = f'https://{url}/api.php?'
    query = (
        f"{base_url}"
        f"action=query&list=categorymembers&cmtitle={urllib.parse.quote(page_name)}"
        f"&cmlimit=max&format=json"
    )
    request = urllib.request.Request(query)
    request.add_header('User-Agent', 'YourAppName/1.0 (s204229@student.dtu.dk)')
    with urllib.request.urlopen(request) as response:
        wikidata = json.loads(response.read())
        return [member['title'] for member in wikidata['query']['categorymembers']]
    
def extract_characters_and_categories(response, downloaded_categories):
    # Words that indicate non-character categories
    '''blacklist = ['places', 'creatures', 'locations', 'objects', 'languages',
                 'rivers', 'mountains', 'events', 'battles', 'wars', 'books',
                 'families', 'songs', 'terms', 'legends', 'real-world', 'media', 
                 'non-canonical', 'images', 'ranks', 'titles']'''
    
    characters = []
    categories = []

    for title in response:
        if title.startswith('Category:'):
            # Normalize for comparison
            lower = title.lower()
            '''if any(bad in lower for bad in blacklist):
                continue  # skip non-character categories'''
            if title not in downloaded_categories:
                categories.append(title)
        else:
            # Keep only likely character pages (heuristic: no colon, not "Category:", not image)
            if ':' not in title and not title.lower().startswith('list of'):
                characters.append(title)
    return list(set(characters)), list(set(categories))

def download(url, start_page, output_name):
    page_names  = get_all_characters(url, start_page)
    pages, failed_batches = download_pages(url, page_names)
    print(len(failed_batches))
    while failed_batches:
        print(f'{len(failed_batches)} pages failed, and are being downloaded now')
        pages_b, failed_batches = download_pages(url, failed_batches)
        pages += pages_b
    with open(output_name, "wb") as f:   # 'wb' = write binary
        pickle.dump(pages, f)

In [3]:
url         = 'harrypotter.fandom.com'
category    = 'Category:Individuals'
download(url, category, 'data/Harry_Potter_Wiki_pages.pkl')

27 remaining. Fetching: Category:Individuals by gender
29 remaining. Fetching: Category:Individuals by place of origin
32 remaining. Fetching: Category:Individuals by deed
58 remaining. Fetching: Category:Horcrux destroyers
57 remaining. Fetching: Category:Unforgivable Curse users
56 remaining. Fetching: Category:Plot to steal the Philosopher's Stone participants
56 remaining. Fetching: Category:Disciplinary hearing of Harry Potter participants
55 remaining. Fetching: Category:Time travellers
54 remaining. Fetching: Category:War veterans
55 remaining. Fetching: Category:Ousting of Severus Snape participants
54 remaining. Fetching: Category:Second World War veterans
53 remaining. Fetching: Category:Deities
52 remaining. Fetching: Category:Unidentified individuals
61 remaining. Fetching: Category:Unidentified portraits
60 remaining. Fetching: Category:Unnamed family members
60 remaining. Fetching: Category:The walk of the Qilin attendees
59 remaining. Fetching: Category:Individuals who k

In [4]:
url         = 'lotr.fandom.com'
category    = 'Category:Characters'
download(url, category, 'data/Lord_of_the_rings_Wiki_pages.pkl')


46 remaining. Fetching: Category:Characters in The Adventures of Tom Bombadil
45 remaining. Fetching: Category:Forest-folk
44 remaining. Fetching: Category:Story tellers
43 remaining. Fetching: Category:Queens
47 remaining. Fetching: Category:The Rings of Power characters
51 remaining. Fetching: Category:Precanonical characters
55 remaining. Fetching: Category:Characters in Born of Hope
54 remaining. Fetching: Category:The Two Towers (film) characters
53 remaining. Fetching: Category:Characters in The Shaping of Middle-earth
52 remaining. Fetching: Category:Kings
62 remaining. Fetching: Category:Númenórean Kings
61 remaining. Fetching: Category:Peter Jackson adaptation characters
65 remaining. Fetching: Category:Characters created for Peter Jackson's trilogy
64 remaining. Fetching: Category:Guardians of Middle-earth characters
63 remaining. Fetching: Category:The Fellowship of the Ring (film) characters
62 remaining. Fetching: Category:Kings of Gondor
61 remaining. Fetching: Category:E